# Plan 4 design-oracle feasibility screen

This notebook reads the immutable Phase 2 analysis artifact. It performs no training, reference fitting, Fisher calculation, or artifact repair. The screen asks whether variable speed creates enough *ideal* controller opportunity to justify a predictive experiment.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'mnist_experiment':
    ROOT = ROOT.parent
analysis_root = ROOT / 'cache' / 'mnist_experiment' / 'plan4' / 'analysis'
artifacts = []
for path in analysis_root.glob('phase2__*') if analysis_root.exists() else []:
    if not (path / 'COMPLETED').is_file():
        continue
    candidate = json.loads((path / 'summary.json').read_text())
    if candidate.get('schema_version') == 5:
        artifacts.append((path, candidate))
if not artifacts:
    raise FileNotFoundError('Run plan4_command_center analyze-phase2; no completed schema-v5 artifact exists.')
ARTIFACT, SUMMARY = max(artifacts, key=lambda item: item[0].stat().st_mtime_ns)
print(f'Loaded {ARTIFACT.relative_to(ROOT)}')
print(f"Decision: {SUMMARY['decision']}")

## Gate summary

In [ ]:
summary_rows = []
for name, result in SUMMARY['schedules'].items():
    event = [row for row in result['trajectory'] if row['in_event_window']]
    principal = result['scenarios']['corrected-oracle-full-grid']
    summary_rows.append({
        'schedule': name,
        'peak delta p': max(row['delta_p'] for row in result['trajectory']),
        'max corrected ||dtheta||': max(np.sqrt(row['displacement_squared_corrected']) for row in event),
        'median trace': np.median([row['oracle_trace_estimate'] for row in event]),
        'max raw pi on bounded state': max(row['bounded_state_raw_oracle_pi'] for row in event),
        'max fully unbounded pi': max(row['unbounded_raw_oracle_pi'] for row in event),
        'steps pi > .07': principal['central']['signal_transition_count'],
        'bounded pi range': principal['central']['event_pi_range'],
        'risk reduction': principal['central']['event_relative_risk_reduction'],
        'robust pass': result['robust_pass'],
    })
gate_table = pd.DataFrame(summary_rows).set_index('schedule')
display(gate_table.style.format({
    'peak delta p': '{:.5f}',
    'max corrected ||dtheta||': '{:.4f}',
    'median trace': '{:.1f}',
    'max raw pi on bounded state': '{:.5f}',
    'max fully unbounded pi': '{:.5f}',
    'bounded pi range': '{:.4f}',
    'risk reduction': '{:.2%}',
}))

## Schedules and speed

In [ ]:
colors = {'linear': '#3f6f8f', 'logistic-k8': '#3a8f5c', 'logistic-k16': '#c47728', 'logistic-k32': '#a23b52', 'logistic-k64': '#6b5ca5', 'logistic-k128': '#16858c', 'logistic-k256': '#ba4a00'}
groups = [('Original screen', ['linear', 'logistic-k8', 'logistic-k16', 'logistic-k32']), ('Breaking-point extension', ['linear', 'logistic-k64', 'logistic-k128', 'logistic-k256'])]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for column, (title, names) in enumerate(groups):
    for name in names:
        rows = SUMMARY['schedules'][name]['trajectory']
        axes[0, column].plot([row['transition'] for row in rows], [row['p'] for row in rows], label=name, color=colors[name])
        axes[1, column].plot([row['transition'] for row in rows], [row['delta_p'] for row in rows], label=name, color=colors[name])
    axes[0, column].set(xlabel='Transition', ylabel='$p_t$', title=title)
    axes[1, column].set(xlabel='Transition', ylabel=r'$\Delta p_t$', title=f'{title}: speed')
    axes[0, column].legend(frameon=False)
fig.tight_layout()
plt.show()

## Ideal actuation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for ax, (title, names) in zip(axes, groups):
    for name in names:
        rows = SUMMARY['schedules'][name]['trajectory']
        ax.plot([row['p'] for row in rows], [row['bounded_state_raw_oracle_pi'] for row in rows], label=name, color=colors[name])
    ax.axhline(.05, color='#222222', linestyle='--', linewidth=1.2, label=r'$\pi_{min}=.05$')
    ax.axhline(.07, color='#7a5a00', linestyle=':', linewidth=1.4, label='signal gate $.07$')
    ax.set(xlabel='$p_t$', ylabel=r'Pre-clipping design-oracle $\pi_t$', title=title)
    ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## Signal scale

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, (title, names) in zip(axes, groups):
    for name in names:
        rows = SUMMARY['schedules'][name]['trajectory']
        ax.plot([row['p'] for row in rows], [np.sqrt(row['displacement_squared_corrected']) for row in rows], label=name, color=colors[name])
    ax.set(xlabel='$p_t$', ylabel=r'Corrected $||d\theta_t||$', title=title)
    ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## Sensitivity checks

In [ ]:
sensitivity_rows = []
for name, result in SUMMARY['schedules'].items():
    for scenario_name, scenario in result['scenarios'].items():
        sensitivity_rows.append({
            'schedule': name,
            'scenario': scenario_name,
            'central signal steps': scenario['central']['signal_transition_count'],
            'q97.5 signal steps': scenario['bootstrap']['signal_transition_count']['q975'],
            'central pi range': scenario['central']['event_pi_range'],
            'q97.5 pi range': scenario['bootstrap']['event_pi_range']['q975'],
            'central risk reduction': scenario['central']['event_relative_risk_reduction'],
            'q97.5 risk reduction': scenario['bootstrap']['event_relative_risk_reduction']['q975'],
            'robust pass': scenario['robust_pass'],
        })
sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity.style.format({
    'central pi range': '{:.5f}',
    'q97.5 pi range': '{:.5f}',
    'central risk reduction': '{:.2%}',
    'q97.5 risk reduction': '{:.2%}',
}))

## Interpretation

In [ ]:
reference = SUMMARY['reference_contract']
message = rf"""
**Decision: `{SUMMARY['decision']}`.** None of the logistic schedules creates material ideal actuation under the frozen bounds. Every bounded oracle remains at $\pi_{{min}}=.05$ for every transition, including under raw displacement, coarsened interpolation, and plug-in trace sensitivity. The unconstrained opportunity is also below the predeclared 5% risk gate.

The result is not caused by the paired noise correction: its median share of raw squared movement is below 0.4% for every schedule. It is limited instead by movement relative to the covariance trace. Only {reference['strictly_converged_points_to_p020']} of {reference['total_reference_points_to_p020']} strict reference checkpoints converged, but the conclusion is unchanged across 1,000 nested bootstrap resamples and the deliberately favorable raw-displacement bound. Under the frozen Plan 4 rules, no predictive Phase 3 run is justified.
"""
display(Markdown(message))